In [1]:
import cv2 as cv # openCV 
from pathlib import Path
import numpy as np

kernel_size = 9
file_names = []

for file in Path("../data/raw").iterdir(): 
  if (file.name.endswith(".jpeg")):
     file_names.append(file.name)
  
for file in file_names:
  image = cv.imread(f"../data/raw/{file}")
  grayscale = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
  blur = cv.GaussianBlur(grayscale, (kernel_size, kernel_size), 3)

  # params: image (post gaussian blur), lower threshold, upper threshold
  # upper threshold = minimum required to be an edge
  # low threshold = minimum required to be an edge, assuming it touches an existing edge (defined by high threshold)
  edges = cv.Canny(blur, 50, 100)

  # params (don't touch first 3)
  # threshold = min number of votes to draw line
  # minLineLength = get rid of segments shorter than that
  # maxLineGap = max gap between colinear segments to be merged into a line
  lines = cv.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=20, maxLineGap=150)

  if lines is not None:
      for line in lines:
          x1, y1, x2, y2 = line
          cv.line(image, (x1, y1), (x2, y2), (255, 0, 255), 3)



  cv.imwrite(f"../data/processed/processed_{file}", image)



In [ ]:
from collections import defaultdict
import cv2 as cv 
import numpy as np 

image = cv.imread("../data/raw/test.jpeg")
blur = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
blur = cv.GaussianBlur(blur, (9, 9), 3)
edges = cv.Canny(blur, 50, 100)
lines = cv.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=20, maxLineGap=150)
horizontal_lines = []
vertical_lines = []
corners = []

for item in lines:
  x1, y1, x2, y2 = item
  if x2 - x1 == 0: 
    pass
  else:
    slope = (y2 - y1) / (x2 - x1)
  if abs(slope) < 0.2:
    horizontal_lines.append([x1, y1, x2, y2])
  if abs(slope) > 20:
     vertical_lines.append([x1, y1, x2, y2])


# if horizontal_lines is not None:
#     for line in horizontal_lines:
#         x1, y1, x2, y2 = line
#         cv.line(image, (x1, y1), (x2, y2), (255, 0, 255), 3)

# if vertical_lines is not None:
#     for line in vertical_lines:
#         x1, y1, x2, y2 = line
#         cv.line(image, (x1, y1), (x2, y2), (255, 0, 255), 3)

# finding corners
def find_line_intersection(line1, line2):
  x1, y1, x2, y2 = map(float, line1)
  x3, y3, x4, y4 = map(float, line2)

    # Ax + By = C
  A1 = y2 - y1
  B1 = x1 - x2
  C1 = A1 * x1 + B1 * y1

  A2 = y4 - y3
  B2 = x3 - x4
  C2 = A2 * x3 + B2 * y3

  # determinant
  det = A1 * B2 - A2 * B1

  if det == 0: 
     return (0,0)

  # intersection
  x = (B2 * C1 - B1 * C2) / det
  y = (A1 * C2 - A2 * C1) / det

  buffer = 10
  lowestx = min(x1, x2)
  lowestx = min(x, lowestx - buffer)
  highestx = max(x1, x2)
  highestx = max(x, highestx + buffer)
  if (x == highestx or x == lowestx):
     return (0,0)

  lowesty = min(y1, y2)
  lowesty = min(y, lowesty - buffer)
  highesty = max(y1, y2)
  highesty = max(y, highesty + buffer)
  if (y == highesty or y == lowesty):
     return (0,0)

  return (x, y)

for line in vertical_lines:
   for line2 in horizontal_lines: 
      x, y = find_line_intersection(line, line2)
      if (x != 0 and y!= 0):
        corners.append((round(x),round(y))) 

from sklearn.cluster import DBSCAN

pts = np.asarray(corners)
labels = DBSCAN(eps=75, min_samples=1).fit_predict(pts)
# for x, y in corners: 
   # cv.circle(image2, (int(x), int(y)), 5, (0, 0, 255), -1)

# cv.imwrite("../data/processed/TESTING2_test.jpeg", image2)

new_corners = np.array([pts[labels == k].mean(axis=0) for k in np.unique(labels)])
for x, y in new_corners: 
   cv.circle(image, (int(x), int(y)), 5, (255, 0, 0), -1)

cv.imwrite("../data/processed/TESTING_test.jpeg", image)
test = new_corners
test.sort()


[[ 792.16666667 1784.16666667]
 [1455.14285714 1796.33333333]
 [ 576.28571429 1781.        ]
 [1231.6        1791.5       ]
 [1010.66666667 1788.        ]
 [ 355.2        1777.        ]
 [1126.         1466.        ]
 [1124.33333333 1243.        ]
 [1126.         1681.33333333]
 [ 901.         1688.16666667]
 [ 901.         1469.85714286]
 [ 901.         1246.6       ]
 [ 900.5        1026.        ]
 [1458.71428571 1572.42857143]
 [1235.6        1568.66666667]
 [1014.8        1565.45      ]
 [ 783.83333333 2332.16666667]
 [1675.16666667 2363.5       ]
 [1446.14285714 2355.85714286]
 [1222.4        2348.        ]
 [1001.         2340.        ]
 [ 806.09090909  902.18181818]
 [2139.75757576 2277.87878788]
 [1906.         2270.52941176]
 [2253.53333333 2281.73333333]
 [2287.         2538.        ]
 [ 785.5        2232.58333333]
 [ 571.         2225.52380952]
 [1003.05882353 2237.05882353]
 [ 349.         2218.5       ]
 [1675.13333333 2262.26666667]
 [1678.16666667 2031.5       ]
 [1451.1